# Build a Proper ML Pipeline with Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

### Load the Raw Data (Not Pre-Encoded)

In [2]:
df = pd.read_csv('Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Basic Cleaning

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

df = df.drop(columns=['customerID'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

### Engineer 2 New Features

Feature 1 — Total number of add-on services subscribed (customers with more services may behave differently):

In [4]:
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                 'StreamingTV', 'StreamingMovies']

def count_services(row):
    return sum(1 for col in service_cols if row[col] not in ['No', 'No phone service', 'No internet service'])

df['TotalServices'] = df.apply(count_services, axis=1)

### Feature 2 — Average charge per month of tenure (captures spending intensity relative to how long they've stayed):

In [5]:
df['ChargePerTenure'] = df['TotalCharges'] / (df['tenure'] + 1)  # +1 avoids divide-by-zero for tenure=0

In [6]:
df[['TotalServices', 'ChargePerTenure']].describe()

,TotalServices,ChargePerTenure
count,7043.000000,7043.000000
mean,4.146244,61.173413
std,2.312720,61.019723
min,1.000000,9.183333
25%,2.000000,26.274411
50%,4.000000,61.150000
75%,6.000000,84.940047
max,9.000000,1397.475000


## Engineered Features
- **TotalServices**: Counts how many add-on services (security, streaming,
  tech support, etc.) a customer subscribes to. Customers with more services
  may be more invested/loyal, so this could correlate with lower churn.
- **ChargePerTenure**: Total charges divided by tenure, capturing spending
  intensity relative to customer lifetime. High values on short tenure may
  signal price shock leading to churn.

### Split Features and Target

In [8]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Identify Numerical vs Categorical Columns

In [9]:
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)

Numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'ChargePerTenure']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\Admin\AppData\Local\Temp\ipykernel_22448\864709933.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


### Build the ColumnTransformer

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
    ]
)

### Build the Full Pipeline (Preprocessing + Model)

In [11]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

### Fit and Evaluate the Pipeline

In [12]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Pipeline Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Pipeline Accuracy: 0.8112136266855926
              precision    recall  f1-score   support

           0       0.84      0.91      0.88      1035
           1       0.68      0.54      0.60       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.74      1409
weighted avg       0.80      0.81      0.80      1409



### Test Whether the Engineered Features Actually Help

In [13]:
X_no_engineered = X.drop(columns=['TotalServices', 'ChargePerTenure'])
X_train_ne, X_test_ne, y_train_ne, y_test_ne = train_test_split(
    X_no_engineered, y, test_size=0.2, random_state=42, stratify=y
)

numerical_cols_ne = X_no_engineered.select_dtypes(include=np.number).columns.tolist()
categorical_cols_ne = X_no_engineered.select_dtypes(include='object').columns.tolist()

preprocessor_ne = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols_ne),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols_ne)
    ]
)

pipeline_ne = Pipeline(steps=[
    ('preprocessor', preprocessor_ne),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_ne.fit(X_train_ne, y_train_ne)
y_pred_ne = pipeline_ne.predict(X_test_ne)

print("Accuracy WITHOUT engineered features:", accuracy_score(y_test_ne, y_pred_ne))
print("Accuracy WITH engineered features:", accuracy_score(y_test, y_pred))

C:\Users\Admin\AppData\Local\Temp\ipykernel_22448\2147028088.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_ne = X_no_engineered.select_dtypes(include='object').columns.tolist()


Accuracy WITHOUT engineered features: 0.8055358410220014
Accuracy WITH engineered features: 0.8112136266855926


In [14]:
joblib.dump(pipeline, 'churn_pipeline.pkl')
print("Pipeline saved as churn_pipeline.pkl")

Pipeline saved as churn_pipeline.pkl


## Do the Engineered Features Help?
Accuracy without engineered features: 80.55%
Accuracy with engineered features: 81.12%

Adding `TotalServices` and `ChargePerTenure` improved accuracy slightly / made
no meaningful difference / slightly hurt performance], suggesting that [these
features carry real signal about churn behavior / the model already captures
this information through existing columns like tenure and service type.